In [1]:
import os
os.chdir(os.environ['PWD'])

import numpy as np
import yaml
from utilities.utils import correct_type_of_entry, get_exp_file_name, create_all_configs, get_min_max_loss
import json
import pandas as pd
from itertools import product
import os

/Users/mathieubazinet/Documents/PythonProjects/ncp2l/ncp2lEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def slicen(n):
    return (slice(None),)*n


zero_one_loss = True

if zero_one_loss:
    choosing_parameter = "full_disagreement_bound_brute_force"
    # choosing_parameter = "validation_error"
    values_to_fetch = ['complement_error', 'validation_error', 'test_error', 'best_model_complement_error','best_model_test_error',
                       'kl_bound', 'disagreement_brute_force', 'full_disagreement_bound_brute_force', 'message_len', 'intrinsic_dim']
else:
    choosing_parameter = "full_disagreement_loss"
    # choosing_parameter = "validation_loss"
    values_to_fetch = ['complement_loss', 'validation_loss', 'test_loss', 'best_model_complement_loss', 'best_model_test_loss','CE_kl_bound',
                       'disagreement_loss_kl', 'full_disagreement_loss', 'message_len']


In [ ]:
results_list = []
dataset_list = []

for dataset_ in ["mnist", 'cifar10']:
    dataset_config_name = "./configs/dataset_configs/" + dataset_ + ".yaml"
    with open(dataset_config_name) as file:
        dataset_configuration = yaml.safe_load(file)

    sweep_config_name = "./configs/experiment_configs/pactl/" + dataset_ + ".yaml"
    with open(sweep_config_name) as file:
        sweep_configuration = yaml.safe_load(file)
    
    hps = {}
    for key, item in sweep_configuration['parameters'].items():
        if item.get('values', None) is not None:
            hps[key] = correct_type_of_entry(item['values'])
    size_hyperparams = tuple([len(l) for l in hps.values()])
    
    results_mat_3 = np.ones(((len(values_to_fetch),) + size_hyperparams))
    results_mat_3[:] = np.nan
    
    for sweep_config in create_all_configs(sweep_configuration):
        file_name = get_exp_file_name(sweep_config|dataset_configuration, path="./pactl_logs/")
        # print(file_name)
        if os.path.exists(file_name):
            with open(file_name) as f:
                config = json.load(f)
                for val_to_fetch_idx in range(len(values_to_fetch)):
                    matrix_idx = tuple([val_to_fetch_idx] + [hps[key].index(config['config'].get(key, None)) for key in hps.keys()])
                    val_to_fetch = values_to_fetch[val_to_fetch_idx]
                    if val_to_fetch == 'intrinsic_dim':
                        results_mat_3[matrix_idx] = config['config'].get(val_to_fetch, None)
                    else:
                        results_mat_3[matrix_idx] = config.get(val_to_fetch, None)

    hp_list = list(hps.values())[1:]
    params_product = list(product(*hp_list))
    name_list = []
    # idx_list = []
    for params in params_product:
        name = ""
        for p in params:
            name += str(p) + " "
        name_list.append(name[:-1])

    if zero_one_loss:
        full_val = values_to_fetch.index("full_disagreement_bound_brute_force")
        kl_val = values_to_fetch.index("kl_bound")
        disag_val = values_to_fetch.index("disagreement_brute_force")
        
        results_mat_3[(full_val,) + slicen(len(results_mat_3.shape)-1)] = results_mat_3[(kl_val,) \
            + slicen(len(results_mat_3.shape)-1)] + results_mat_3[(disag_val,) + slicen(len(results_mat_3.shape)-1)]
    
    reshaped_matrix = np.nanmean(results_mat_3, axis=1).reshape(results_mat_3.shape[0],np.prod(results_mat_3.shape[2:])).T
    mean_df = pd.DataFrame(reshaped_matrix, index=name_list, columns=values_to_fetch)
    
    reshaped_matrix_std = np.nanstd(results_mat_3, axis=1).reshape(results_mat_3.shape[0],np.prod(results_mat_3.shape[2:])).T
    std_df = pd.DataFrame(reshaped_matrix_std, index=name_list, columns=values_to_fetch)
    
    if "model_type" in list(hps.keys()):
        list_indexs = [[] for _ in range(len(hps['model_type']))]
        for name in list(mean_df.index):
            for model_idx in range(len(hps['model_type'])):
                if hps['model_type'][model_idx] in name:
                    list_indexs[model_idx].append(name)
                    break
    else:
        list_indexs = [list(mean_df.index)]
    
    for i in range(len(list_indexs)):
        index_mean = mean_df.loc[list_indexs[i]].sort_values(by=choosing_parameter).index[0]
        
        best_datapoint_mean = mean_df.loc[index_mean]
        best_datapoint_std = std_df.loc[index_mean]

        if dataset_ == "mnist":
            mt = "CNN"
        elif dataset_ == "cifar10":
            mt = "ResNet18"
    
        rounding_val = 2 if zero_one_loss else 4
        rounding = f"%.{rounding_val}f"
        
        mult = 100.0 if zero_one_loss else 1.0
        
        cpr = (rounding % round(best_datapoint_mean['message_len']/8000,rounding_val)) + r"$\pm$"
        cpr += (rounding % round(best_datapoint_std['message_len']/8000,rounding_val))
    
        if zero_one_loss:
            vals_to_add = ["Model", 'complement_error', 'test_error', "cpr", 'best_model_complement_error', 'best_model_test_error',
                            'kl_bound', 'disagreement_brute_force', 'full_disagreement_bound_brute_force', 'intrinsic_dim']
        else:
            vals_to_add = ["Model", 'complement_loss', 'test_loss', "cpr", 'best_model_complement_loss',
                           'best_model_test_loss', 'CE_kl_bound', 'disagreement_loss_kl', 'full_disagreement_loss', 'intrinsic_dim']
    
        list_of_vals = []
        for val in vals_to_add:
            if val == "Model":
                list_of_vals.append(mt)
                continue
            elif val == "cpr":
                list_of_vals.append(cpr)
                continue
            elif val == 'intrinsic_dim':
                list_of_vals.append(str(best_datapoint_mean[val]))
                continue
    
            temp = (rounding % round(mult*best_datapoint_mean[val],rounding_val)) + r"$\pm$"
            temp += (rounding % round(mult*best_datapoint_std[val], rounding_val))
            list_of_vals.append(temp)
    
        results_list.append(pd.Series(list_of_vals, index=vals_to_add))
            
        dataset_list.append(dataset_.upper())

if zero_one_loss:
    print(pd.DataFrame(results_list, index=dataset_list).to_latex(float_format="%.2f"))
else:
    print(pd.DataFrame(results_list, index=dataset_list).to_latex(float_format="%.4f"))

\begin{tabular}{lllllllllll}
\toprule
 & Model & complement_error & test_error & cpr & best_model_complement_error & best_model_test_error & kl_bound & disagreement_brute_force & full_disagreement_bound_brute_force & intrinsic_dim \\
\midrule
MNIST & CNN & 0.93$\pm$0.07 & 0.89$\pm$0.08 & 0.04$\pm$0.00 & 0.29$\pm$0.03 & 0.50$\pm$0.07 & 2.45$\pm$0.13 & 1.00$\pm$0.06 & 3.45$\pm$0.11 & 100.0 \\
CIFAR10 & ResNet18 & 14.30$\pm$0.30 & 14.95$\pm$0.42 & 0.03$\pm$0.00 & 0.00$\pm$0.00 & 5.84$\pm$0.16 & 20.15$\pm$0.35 & 14.91$\pm$0.09 & 35.06$\pm$0.40 & 100.0 \\
\bottomrule
\end{tabular}

